# Gröbner basis — analysis notebook

Inspect dataset and (post-training) model outputs for the Gröbner basis task.

**Prerequisite**: dataset generated and model trained:
```bash
cd ../experiments/toy/scripts
python generate.py
python train.py        # default --training_order degrevlex
```

Patterns mirror `calt-codebase/examples/demos/minimal_demo.ipynb`.

In [ ]:
import os
import sys
from pathlib import Path

_TASK_ROOT = Path.cwd().parent
_REPO_ROOT = _TASK_ROOT.parent
os.chdir(_TASK_ROOT / 'experiments' / 'toy' / 'scripts')
sys.path.insert(0, str(_REPO_ROOT))

from omegaconf import OmegaConf
from calt.dataset.sagemath.utils.polynomial_sampler import PolynomialSampler

from groebner_basis.core.generator import GroebnerGenerator
from shared import showcase

## 1. A few raw samples (F → G) over QQ[x, y]

In [ ]:
data_cfg = OmegaConf.load('../configs/data.yaml')
sampler = PolynomialSampler(**OmegaConf.to_container(data_cfg.sampler, resolve=True))
gen = GroebnerGenerator(sampler=sampler, **OmegaConf.to_container(data_cfg.problem_generator, resolve=True))

for seed in range(3):
    F, G = gen(seed)
    print(f'seed={seed}')
    print(f'  F = {[str(f) for f in F]}')
    print(f'  G = {[str(g) for g in G]}')
    print()

## 2. Stored dataset

In [ ]:
train_path = Path('../data/QQ/train_raw.txt')
if not train_path.exists():
    print(f'No dataset at {train_path.resolve()}. Run scripts/generate.py first.')
else:
    lines = train_path.read_text().strip().split('\n')
    print(f'{len(lines)} train samples. First 3:')
    for line in lines[:3]:
        print(' ', line)

## 3. Showcase eval results — canonical CALT pattern

`run_training` suffixes the output directory with `_{training_order}`. We try
`results_degrevlex` first, then `results_lex`.

In [ ]:
from calt.io import IOPipeline

cfg = OmegaConf.load('../configs/train.yaml')
io_dict = IOPipeline.from_config(cfg.data).build()

candidates = ['../outputs/results_degrevlex', '../outputs/results_lex', '../outputs/results']
results_dir = next((c for c in candidates if Path(c).exists()), None)

if results_dir is None:
    print('No results directory found. Run train.py first.')
else:
    print(f'Using {results_dir}')
    showcase(io_dict['test_dataset'], success_cases=True,  num_show=5, results_dir=results_dir)
    print()
    showcase(io_dict['test_dataset'], success_cases=False, num_show=5, results_dir=results_dir)

## 4. Learning curve

In [ ]:
from shared.plotting import plot_success_rate
import matplotlib.pyplot as plt

if results_dir is not None:
    try:
        ax = plot_success_rate(results_dir)
        plt.show()
    except FileNotFoundError as e:
        print(e)